In [1]:
import numpy as np

In [2]:
# GLOBAL SETTINGS

SEED = 42

In [6]:
# UTILS 

def sample_param(
    value: tuple[float, float],
    rng: np.random.Generator,
    name: str,
) -> float:
    """
    Sample a scalar uniformly from a (min, max) tuple.

    Parameters
    ----------
    value : tuple[float, float]
        Range to sample from.
    rng : np.random.Generator
        NumPy random number generator.
    name : str
        Parameter name for error messages.

    Returns
    -------
    float
        Sampled scalar value.
    """

    if len(value) != 2:
        raise ValueError(f"{name} must be a tuple of length 2")

    low, high = value
    if high < low:
        raise ValueError(f"Invalid range for {name}: {value}")

    return float(rng.uniform(low, high))

def _sample_lengths(
        n_series: int,
        length: tuple[int, int],
        rng: np.random.Generator,
    ) -> np.ndarray:
    """
    Sample integer lengths uniformly from [min_len, max_len]
    """

    min_len, max_len = length
    if min_len <= 0 or max_len < min_len:
        raise ValueError("length must be a tuple (min_len, max_len) with 0 < min_len <= max_len")
    
    return rng.integers(min_len, max_len + 1, size = n_series)

def save_dataset(path: str, series_list: list[np.ndarray]) -> None:
    """
    Save a variable-length list of NumPy arrays to a .npz file.

    Parameters
    ----------
    path : str 
        Output npz file path
    series_list : list[np.ndarray]
        List of arrays, each shaped (T, C), where T may vary across series

    Notes
    -----
    Variable-length arrays are stored as an object array inside the npz.
    """

    obj_array = np.empty(len(series_list), dtype = object)
    for i, arr in enumerate(series_list):
        obj_array[i] = arr

    np.savez(path, series = obj_array)

def load_dataset(path: str) -> list[np.ndarray]:
    """
    Load a dataset saved with save_dataset.
    """

    with np.load(path, allow_pickle = True) as data:
        series = data["series"]
        return list(series)

# Univariate Series

## Sinusoids

Distinguishing between different periodicities

In [4]:
def generate_sinusoid_dataset(
        n_series: int,
        length: tuple[int, int],
        seed: int | None,
        frequency: tuple[float, float],
        amplitude: tuple[float, float],
        vertical_shift: tuple[float, float],
        phase_shift: tuple[float, float],
        slope: tuple[float, float],
        noise_std: tuple[float, float],
        perturbations: dict[str, tuple[float, float]] | None,
    ) -> list[np.ndarray]:
    """
    Generate variable-length univariate sinusoidal time series.

    Parameters
    ----------
    n_series : int
        Number of series to generate.
    length : tuple[int, int]
        Range of lengths as (min_len, max_len). Each series length is sampled uniformly from this interval.
    seed : int or None
        Random seed for reproducibility.
    frequency : tuple[float, float]
        Frequency range in cycles per sample.
    amplitude : tuple[float, float]
        Amplitude range.
    vertical_shift : tuple[float, float]
        Constant vertical offset range.
    phase_shift : tuple[float, float]
        Phase offset range in radians.
    slope : tuple[float, float]
        Linear drift slope range.
    noise_std : tuple[float, float]
        Standard deviation range for additive Gaussian noise.
    perturbations : dict[str, tuple[float, float]] or None
        Optional perturbations. Supported keys are:
            - "amplitude_jitter_strength" - amplitude change over time 
            - "frequency_jitter_strength" - frequency change over time 
            - "baseline_wander_amplitude" - amplitude of additive baseline wander
            - "baseline_wander_frequency" - frequency of additive baseline wander

    Returns
    -------
    list[np.ndarray]
        List of arrays shaped (T, 1).
    """

    if n_series <= 0:
        raise ValueError("n_series must be positive")

    rng = np.random.default_rng(seed)
    lengths = _sample_lengths(n_series, length, rng)

    series_list: list[np.ndarray] = []

    for T in lengths:
        t = np.arange(T, dtype = float)

        # base series parameters and checks 
        freq = sample_param(frequency, rng, "frequency")
        amp = sample_param(amplitude, rng, "amplitude")
        vshift = sample_param(vertical_shift, rng, "vertical_shift")
        phase = sample_param(phase_shift, rng, "phase_shift")
        drift = sample_param(slope, rng, "slope")
        noise = sample_param(noise_std, rng, "noise_std")
        if freq < 0.0:
            raise ValueError("frequency must be nonnegative")
        if amp < 0.0:
            raise ValueError("amplitude must be nonnegative")
        if noise < 0.0:
            raise ValueError("noise_std must be nonnegative")

        amp_envelope = np.ones(T, dtype = float)
        inst_freq = np.full(T, freq, dtype = float)

        # additional perturbations 
        if perturbations is not None:

            # slowly varying sine wave as amplitude envelope 
            # envelope amplitude is user set, shift and frequency are sampled from fixed ranges 
            if "amplitude_jitter_strength" in perturbations:
                amp_mod_strength = sample_param(perturbations["amplitude_jitter_strength"], rng, "amplitude_jitter_strength")
                if amp_mod_strength < 0.0:
                    raise ValueError("amplitude_jitter_strength must be nonnegative")
                amp_mod_freq = rng.uniform(0.005, 0.03)
                amp_mod_phase = rng.uniform(0.0, 2.0 * np.pi)
                amp_envelope = 1.0 + amp_mod_strength * np.sin(2.0 * np.pi * amp_mod_freq * t + amp_mod_phase)

            # slowly varying sine wave added to the base frequency over time
            # jitter strength is user set, while modulation phase and frequency are sampled from fixed ranges
            if "frequency_jitter_strength" in perturbations:
                freq_jitter_strength = sample_param(perturbations["frequency_jitter_strength"], rng, "frequency_jitter_strength")
                if freq_jitter_strength < 0.0:
                    raise ValueError("frequency_jitter_strength must be nonnegative")
                freq_jitter_freq = rng.uniform(0.005, 0.03)
                freq_jitter_phase = rng.uniform(0.0, 2.0 * np.pi)
                inst_freq = freq + freq_jitter_strength * np.sin(2.0 * np.pi * freq_jitter_freq * t + freq_jitter_phase)
                inst_freq = np.clip(inst_freq, 0.0, None)

        inst_phase = phase + 2.0 * np.pi * np.cumsum(inst_freq)
        y = amp * amp_envelope * np.sin(inst_phase)
        y = y + drift * t
        y = y + vshift

        if perturbations is not None:

            # additive baseline variance
            if "baseline_wander_amplitude" in perturbations:
                if "baseline_wander_frequency" not in perturbations:
                    raise ValueError("baseline_wander_frequency must be provided when baseline_wander_amplitude is used")
                
            if "baseline_wander_amplitude" in perturbations:
                baseline_amp = sample_param(perturbations["baseline_wander_amplitude"], rng, "baseline_wander_amplitude")
                baseline_freq = sample_param(perturbations["baseline_wander_frequency"], rng, "baseline_wander_frequency")
                if baseline_amp < 0.0:
                    raise ValueError("baseline_wander_amplitude must be nonnegative")
                if baseline_freq < 0.0:
                    raise ValueError("baseline_wander_frequency must be nonnegative")
                baseline_phase = rng.uniform(0.0, 2.0 * np.pi)
                baseline = baseline_amp * np.sin(2.0 * np.pi * baseline_freq * t + baseline_phase)
                y = y + baseline

        if noise > 0.0:
            y = y + rng.normal(loc = 0.0, scale = noise, size = T)

        series_list.append(y[:, None].astype(np.float32))

    return series_list

In [ ]:
class_1 = generate_sinusoid_dataset(
    n_series = 20000,
    length = (1500, 2000),
    seed = SEED,
    frequency = (0.020, 0.022),
    amplitude = (1.00, 1.03),
    vertical_shift = (0.00, 0.00),
    phase_shift = (0.00, 0.05),
    slope = (0.00, 0.00),
    noise_std = (0.008, 0.015),
    perturbations = {
        "amplitude_jitter_strength": (0.015, 0.035),
        "frequency_jitter_strength": (0.0003, 0.0008),
        "baseline_wander_amplitude": (0.010, 0.030),
        "baseline_wander_frequency": (0.0003, 0.0010),
    },
)

class_2 = generate_sinusoid_dataset(
    n_series = 20000,
    length = (1500, 2000),
    seed = SEED,
    frequency = (0.032, 0.034),
    amplitude = (1.18, 1.22),
    vertical_shift = (0.00, 0.00),
    phase_shift = (0.00, 0.05),
    slope = (0.00, 0.00),
    noise_std = (0.008, 0.015),
    perturbations = {
        "amplitude_jitter_strength": (0.015, 0.035),
        "frequency_jitter_strength": (0.0003, 0.0008),
        "baseline_wander_amplitude": (0.010, 0.030),
        "baseline_wander_frequency": (0.0003, 0.0010),
    },
)

class_3 = generate_sinusoid_dataset(
    n_series = 20000,
    length = (1500, 2000),
    seed = SEED,
    frequency = (0.020, 0.022),
    amplitude = (1.00, 1.03),
    vertical_shift = (0.00, 0.00),
    phase_shift = (0.00, 0.05),
    slope = (0.00018, 0.00024),
    noise_std = (0.008, 0.015),
    perturbations = {
        "amplitude_jitter_strength": (0.015, 0.035),
        "frequency_jitter_strength": (0.0003, 0.0008),
        "baseline_wander_amplitude": (0.010, 0.030),
        "baseline_wander_frequency": (0.0003, 0.0010),
    },
)

class_4 = generate_sinusoid_dataset(
    n_series = 20000,
    length = (1500, 2000),
    seed = SEED,
    frequency = (0.020, 0.022),
    amplitude = (1.00, 1.03),
    vertical_shift = (0.00, 0.00),
    phase_shift = (0.00, 0.05),
    slope = (-0.00024, -0.00018),
    noise_std = (0.008, 0.015),
    perturbations = {
        "amplitude_jitter_strength": (0.015, 0.035),
        "frequency_jitter_strength": (0.0003, 0.0008),
        "baseline_wander_amplitude": (0.010, 0.030),
        "baseline_wander_frequency": (0.0003, 0.0010),
    },
)

class_5 = generate_sinusoid_dataset(
    n_series = 20000,
    length = (1500, 2000),
    seed = SEED,
    frequency = (0.020, 0.022),
    amplitude = (1.00, 1.03),
    vertical_shift = (0.30, 0.36),
    phase_shift = (0.00, 0.05),
    slope = (0.00, 0.00),
    noise_std = (0.008, 0.015),
    perturbations = {
        "amplitude_jitter_strength": (0.015, 0.035),
        "frequency_jitter_strength": (0.0003, 0.0008),
        "baseline_wander_amplitude": (0.010, 0.030),
        "baseline_wander_frequency": (0.0003, 0.0010),
    },
)

all_series = class_1 + class_2 + class_3 + class_4 + class_5

save_dataset("c.npz", all_series)

labels = [1]*20000 + [2]*20000 + [3]*20000 + [4]*20000 + [5]*20000
labels = np.asarray(labels)
np.savetxt('labels.csv', labels, delimiter = ',')

## Autoregressive processes

Distinguishing between different degrees of temporal dependence

## Regime Switching

Distinguishing between latent generative processes

## Burst/spike processes

Anomaly detection

## Shapelet-based series

Build sequences from reusable motifs inserted at random positions

# Multivariate Series

## Shared latent factors

Distinguish between known low-dimensional latent structures

## Lagged dependencies

## State-space models